<a href="https://colab.research.google.com/github/4b654/LAB-1/blob/main/amharic_spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio

In [ ]:
import pandas as pd
import string
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import gradio as gr
import numpy as np
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Data Preparation
class SpamDataset(Dataset):
    def __init__(self, texts, labels, vectorizer):
        self.texts = texts
        self.labels = labels
        self.vectorizer = vectorizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        vector = self.vectorizer.transform([text]).toarray().squeeze()
        vector = torch.FloatTensor(vector)
        label = torch.FloatTensor([1]) if label == 'spam' else torch.FloatTensor([0])
        return vector, label

# 2. Model Architecture
class SpamClassifier(nn.Module):
    def __init__(self, input_dim):
        super(SpamClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

# 3. Preprocessing Function
def preprocess_text(text):
    # Define stopwords
    english_stopwords = set(stopwords.words('english'))
    amharic_stopwords = [
        "እና", "ከ", 'እንዲህ', 'እንደ', 'እንደምን',
        'እንዴት', 'እንደዚህ', 'እንደው', 'እንድ',
        'እንድህ', 'እንድምን', 'እንድድ', 'እንድዚህ',
        'እንድው', 'እንድን', 'እንድንህ', 'እንድንምን',
        'እንድንድ', 'እንድንዚህ', 'እንድንው'
    ]
    all_stopwords = english_stopwords.union(set(amharic_stopwords))

    amharic_punctuation = [
        '።', '፣', '፤', '፨', '፠', '`', '^', '!', '፡',
        "፦", "፥", "፡-፡", "-", "<", ">", "...", "?", "/", "+", "*","(",")"
    ]

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation + ''.join(amharic_punctuation)))

    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word.lower() not in all_stopwords]

    return " ".join(tokens)

# 4. Training Function
def train_model(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for vectors, labels in dataloader:
        vectors = vectors.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(vectors)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

# 5. Evaluation Function
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for vectors, labels in dataloader:
            vectors = vectors.to(device)
            labels = labels.to(device)

            outputs = model(vectors)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

# 6. Main Execution
if __name__ == "__main__":
    # Load and preprocess data
    data = pd.read_csv('/content/spamdetector_dataset.csv')
    data["cleaned_text"] = data[" message"].apply(preprocess_text)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        data["cleaned_text"], data["label"], test_size=0.15, random_state=42
    )

    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(max_features=1000)
    vectorizer.fit(X_train)

    # Create datasets and dataloaders
    train_dataset = SpamDataset(X_train.values, y_train.values, vectorizer)
    test_dataset = SpamDataset(X_test.values, y_test.values, vectorizer)

    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    input_dim = len(vectorizer.get_feature_names_out())
    model = SpamClassifier(input_dim).to(device)

    # Loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    num_epochs = 10
    for epoch in range(num_epochs):
        train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_model(model, test_loader, criterion, device)

        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    # 7. Prediction Function for Gradio
    def predict_spam(message, threshold=0.25):
        cleaned_message = preprocess_text(message)
        vector = vectorizer.transform([cleaned_message]).toarray().squeeze()
        vector = torch.FloatTensor(vector).unsqueeze(0).to(device)

        with torch.no_grad():
            model.eval()
            prob = model(vector).item()

        if prob >= threshold:
            return f"spam(የመሆን እድሉ: {prob*100:.2f}%)"
        else:
            return f"not spam (የመሆን እድሉ: {prob*100:.2f}%)"

    # 8. Create Gradio Interface
    iface = gr.Interface(
        fn=predict_spam,
        inputs="text",
        outputs="text",
        title="Amharic Spam Detector (PyTorch)",
        description="Enter an Amharic message to check if it's spam"
    )

    # Launch the interface
    iface.launch()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Epoch [1/10], Train Loss: 0.6721, Train Acc: 70.10%, Val Loss: 0.6274, Val Acc: 80.86%
Epoch [2/10], Train Loss: 0.5343, Train Acc: 88.30%, Val Loss: 0.4529, Val Acc: 88.28%
Epoch [3/10], Train Loss: 0.3271, Train Acc: 95.02%, Val Loss: 0.3033, Val Acc: 91.80%
Epoch [4/10], Train Loss: 0.1973, Train Acc: 96.47%, Val Loss: 0.2360, Val Acc: 93.36%
Epoch [5/10], Train Loss: 0.1344, Train Acc: 97.37%, Val Loss: 0.2051, Val Acc: 93.36%
Epoch [6/10], Train Loss: 0.1011, Train Acc: 98.06%, Val Loss: 0.1910, Val Acc: 94.14%
Epoch [7/10], Train Loss: 0.0842, Train Acc: 98.20%, Val Loss: 0.1815, Val Acc: 94.92%
Epoch [8/10], Train Loss: 0.0674, Train Acc: 98.48%, Val Loss: 0.1762, Val Acc: 95.31%
Epoch [9/10], Train Loss: 0.0578, Train Acc: 98.96%, Val Loss: 0.1732, Val Acc: 95.70%
Epoch [10/10], Train Loss: 0.0499, Train Acc: 98.96%, Val Loss: 0.1705, Val Acc: 96.09%
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True

In [ ]:
!pip install streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import string
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import gradio as gr
import joblib

# Download stopwords
nltk.download("stopwords")
from nltk.corpus import stopwords

# Define stopwords
english_stopwords = set(stopwords.words('english'))
amharic_stopwords = [
    "እና", "ከ", 'እንዲህ', 'እንደ', 'እንደምን',
    'እንዴት', 'እንደዚህ', 'እንደው', 'እንድ',
    'እንድህ', 'እንድምን', 'እንድድ', 'እንድዚህ',
    'እንድው', 'እንድን', 'እንድንህ', 'እንድንምን',
    'እንድንድ', 'እንድንዚህ', 'እንድንው'
]
# Combining stopwords
all_stopwords = english_stopwords.union(set(amharic_stopwords))

# Load dataset
data = pd.read_csv("/content/spamdetector_dataset.csv")

# Preprocess text
def preprocess_text(text):
    amharic_punctuation = [
        '።', '፣', '፤', '፨', '፠', '`', '^', '!', '፡',
        "፦", "፥", "፡-፡", "-", "<", ">", "...", "?", "/", "+", "*"
    ]

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation + ''.join(amharic_punctuation)))

    return text

    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word.lower() not in all_stopwords]

    return " ".join(tokens)

# Apply preprocessing
data["cleaned_text"] = data[" message"].apply(preprocess_text)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data["cleaned_text"], data["label"], test_size=0.2, random_state=1348
)

# Vectorize text
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Save model and vectorizer
joblib.dump(model, 'amharic_spam_detector.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# Predict function with threshold
def predict_spam(message, threshold=50):
    cleaned_message = preprocess_text(message)
    message_tfidf = vectorizer.transform([cleaned_message])
    prediction_prob = model.predict_proba(message_tfidf)[0][1] * 100

    if prediction_prob >= threshold:
        return f"ሀሰተኛ መልእክት (የመሆን እድል: {prediction_prob:2f}%)"
    else:
        return f"እውነተኛ መልእክት (የመሆን እድል: {prediction_prob:2f}%)"

# Gradio interface
iface = gr.Interface(
    fn=predict_spam,
    inputs="text",
    outputs="text",
    title="AMHARIC SPAM DETECTOR",
    description="Enter a message to check if it is spam."
)

# Function to calculate performance metrics
def calculate_performance_metrics():
    pred = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, pos_label='spam', average='binary')
    recall = recall_score(y_test, pred, pos_label='spam', average='binary')
    f1 = f1_score(y_test, pred, pos_label='spam', average='binary')

    return {
        "Accuracy": accuracy * 100,
        "Precision": precision * 100,
        "Recall": recall * 100,
        "F1 Score": f1 * 100
    }

if __name__ == "__main__":
    iface.launch()
    metrics = calculate_performance_metrics()
    print(f"Model Performance Metrics: {metrics}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9274fe81567794ca0c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Model Performance Metrics: {'Accuracy': 95.17241379310344, 'Precision': 93.97590361445783, 'Recall': 97.5, 'F1 Score': 95.70552147239265}


In [ ]:
import pandas as pd
data=pd.read_csv('/content/spamdetector_dataset.csv')
print(data)

     label                                            message
0     spam     የእርስዎ የበረራ ቦታ ማስያዝ አልተሳካም። በቅናሽ እንደገና ለማስያዝ...
1     spam     ከአዲስ መሣሪያ የመግባት ሙከራ አግኝተናል። የጂሜይል መለያህን አሁን...
2      ham                                 ሄይ፣  ቀንህ እንዴት ነው? 
3      ham                        ነገ በ1 ሰዓት የምሳ እቅዳችንን አትርሱ! 
4     spam     1000 ዶላር የስጦታ ካርድ አሸንፈሃል! አሁን ሽልማትዎን ለማግኘት ...
...    ...                                                ...
1188  spam  pdate_Now - Double mins and 1000 txts on Orang...
1189  spam  Ur cash-balance is currently 500 pounds - to m...
1190  spam  URGENT! Your Mobile number has been awarded wi...
1191  spam  Sorry! U can not unsubscribe yet. THE MOB offe...
1192  spam   You have 1 new message. Please call 08712400200.

[1193 rows x 2 columns]


In [ ]:
!pip install gradio_client

In [ ]:
import pandas as pd
import string
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import gradio as gr
import joblib
from gradio_client import Client

# Download stopwords
nltk.download("stopwords")
from nltk.corpus import stopwords

# Define stopwords
english_stopwords = set(stopwords.words('english'))
amharic_stopwords = [
    "እና", "ከ", 'እንዲህ', 'እንደ', 'እንደምን',
    'እንዴት', 'እንደዚህ', 'እንደው', 'እንድ',
    'እንድህ', 'እንድምን', 'እንድድ', 'እንድዚህ',
    'እንድው', 'እንድን', 'እንድንህ', 'እንድንምን',
    'እንድንድ', 'እንድንዚህ', 'እንድንው'
]
# Combining stopwords
all_stopwords = english_stopwords.union(set(amharic_stopwords))

# Load dataset
data = pd.read_csv("spamdetector_dataset.csv")

# Preprocess text
def preprocess_text(text):
    amharic_punctuation = [
        '።', '፣', '፤', '፨', '፠', '`', '^', '!', '፡',
        "፦", "፥", "፡-፡", "-", "<", ">", "...", "?", "/", "+", "*"
    ]

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation + ''.join(amharic_punctuation)))

    return text

    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word.lower() not in all_stopwords]

    return " ".join(tokens)

# Apply preprocessing
data["cleaned_text"] = data[" message"].apply(preprocess_text)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data["cleaned_text"], data["label"], test_size=0.2, random_state=42
)

# Vectorize text
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Save model and vectorizer
joblib.dump(model, 'amharic_spam_detector.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# Predict function with threshold
def predict_spam(message, threshold=50):
    cleaned_message = preprocess_text(message)
    message_tfidf = vectorizer.transform([cleaned_message])
    prediction_prob = model.predict_proba(message_tfidf)[0][1] * 100

    if prediction_prob >= threshold:
        return f"Spam (probability: {prediction_prob:.2f}%)"
    else:
        return f"Not Spam (probability: {prediction_prob:.2f}%)"

# Gradio interface
iface = gr.Interface(
    fn=predict_spam,
    inputs="text",
    outputs="text",
    title="SPAM DETECTOR",
    description="Enter a message to check if it is spam."

)

# Function to calculate performance metrics
def calculate_performance_metrics():
    pred = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, pos_label='spam', average='binary')
    recall = recall_score(y_test, pred, pos_label='spam', average='binary')
    f1 = f1_score(y_test, pred, pos_label='spam', average='binary')

    return {
        "Accuracy": accuracy * 100,
        "Precision": precision * 100,
        "Recall": recall * 100,
        "F1 Score": f1 * 100
    }

if __name__ == "__main__":
    iface.launch()
    metrics = calculate_performance_metrics()
    print(f"Model Performance Metrics:.{metrics}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30efee3515217a425a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Model Performance Metrics:.{'Accuracy': 94.48275862068965, 'Precision': 92.7710843373494, 'Recall': 97.46835443037975, 'F1 Score': 95.06172839506173}


In [ ]:
import pandas as pd
import string
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import gradio as gr
import joblib
import matplotlib.pyplot as plt

# Download stopwords
nltk.download("stopwords")
from nltk.corpus import stopwords

# Define stopwords
english_stopwords = set(stopwords.words('english'))
amharic_stopwords = [
    "እና", "ከ", 'እንዲህ', 'እንደ', 'እንደምን',
    'እንዴት', 'እንደዚህ', 'እንደው', 'እንድ',
    'እንድህ', 'እንድምን', 'እንድድ', 'እንድዚህ',
    'እንድው', 'እንድን', 'እንድንህ', 'እንድንምን',
    'እንድንድ', 'እንድንዚህ', 'እንድንው'
]
# Combining stopwords
all_stopwords = english_stopwords.union(set(amharic_stopwords))

# Load dataset
data = pd.read_csv('/content/spamdetector_dataset.csv')

# Preprocess text
def preprocess_text(text):
    amharic_punctuation = [
        '።', '፣', '፤', '፨', '፠', '`', '^', '!', '፡',
        "፦", "፥", "፡-፡", "-", "<", ">", "...", "?", "/", "+", "*", "(", ")"
    ]

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation + ''.join(amharic_punctuation)))

    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word.lower() not in all_stopwords]

    return " ".join(tokens)

# Apply preprocessing
data["cleaned_text"] = data[" message"].apply(preprocess_text)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data["cleaned_text"], data["label"], test_size=0.15, random_state=21
)

# Vectorize text
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Save model and vectorizer
joblib.dump(model, 'amharic_spam_detector.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# Predict function with threshold and pie chart
def predict_spam(message, threshold=25):
    cleaned_message = preprocess_text(message)
    message_tfidf = vectorizer.transform([cleaned_message])
    prediction_prob = model.predict_proba(message_tfidf)[0][1] * 100
    not_spam_prob = 100 - prediction_prob

    # Create pie chart
    labels = ['Spam', 'Not Spam']
    sizes = [prediction_prob, not_spam_prob]
    colors = ['#ff9999','#66b3ff']
    explode = (0.1, 0)  # explode the 1st slice (Spam)

    plt.figure(figsize=(6, 6))
    plt.pie(sizes, explode=explode, labels=labels, colors=colors,
            autopct='%1.1f%%', shadow=True, startangle=90)
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    plt.title('Spam Prediction Probability')
    plt.show()

    if prediction_prob >= threshold:
        return f"ሀሰተኛ መልእክት (የመሆን እድል: {prediction_prob:.2f}%)"
    else:
        return f"እውነተኛ መልእክት (የመሆን እድል: {prediction_prob:.2f}%)"

# Gradio interface
iface = gr.Interface(
    fn=predict_spam,
    inputs="text",
    outputs="text",
    title="Amharic Spam Detector",
    description="Enter a message to check if it is spam."
)

if __name__ == "__main__":
    iface.launch()

# Function to calculate accuracy
def calculate_accuracy():
    pred = model.predict(X_test_tfidf)
    score = accuracy_score(y_test, pred)
    return score * 100

print(f"Model Accuracy: {calculate_accuracy():.2f}%")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90fc5cf9fb1980a042.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Model Accuracy: 95.41%
